## Data Ingestion

In [21]:
### Document Datastructure

from langchain_core.documents import Document

In [22]:
doc = Document(
    page_content = "this is the main content I am using to create RAG",
    metadata = {
        "source":"example.txt",
        "pages":1,
        "author":"Mathura Santhalingam",
        "date_created":"20/08/2026"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Mathura Santhalingam', 'date_created': '20/08/2026'}, page_content='this is the main content I am using to create RAG')

In [23]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)


In [24]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
     
    """
}
for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print(" Sample text files created.")

 Sample text files created.


In [25]:
### Text Loader (how to read the text above using TextLoader)
from langchain.document_loaders import TextLoader
# OR : from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")

# loader is the act of reading, it will give an object of Textloader
document=loader.load() # to get the content inside it
print(document)


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [26]:
### Directory Loader(how to read the text above using DirectoryLoader, i want to read the text when i have all the important files in my directory)
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader = DirectoryLoader(
    "../data/text_files", 
    glob="**/*.txt", ## pattern to match files on the files
    loader_cls= TextLoader, ##loader class to use, which file we plan to load, if pdf, write pdf instead of TextLoaders
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'),
 Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervise

In [27]:
### for PDF
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader = DirectoryLoader(
    "../data/pdf/", 
    glob="**/*.pdf",
    loader_cls= PyMuPDFLoader,
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'macOS Version 26.5.2 (assemblage 25F84) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20260819230533Z00'00'", 'source': '../data/pdf/def.pdf', 'file_path': '../data/pdf/def.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'def', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260819230533Z00'00'", 'trapped': '', 'modDate': "D:20260819230533Z00'00'", 'creationDate': "D:20260819230533Z00'00'", 'page': 0}, page_content="Dans une architecture RAG (Retrieval-Augmented Generation), l'ingestion des données constitue la \npremière étape critique pour enrichir le contexte d'un LLM avec des sources externes. \nLe rôle des loaders \nLes document loaders servent de passerelle entre les formats d'origine et le pipeline de traitement. \nQu'il s'agisse de fichiers locaux (PDF, Markdown, CSV), de pages web ou de bases de données \n(Notion, SQL), le chargeur extrait le contenu brut, nettoie les données non pertinentes et standardise \nle f

In [28]:
type(pdf_documents[0]) ###as the above is a list of documents

langchain_core.documents.base.Document

### Chunking

In [29]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [30]:
chunks=split_documents(pdf_documents)
chunks

Split 1 documents into 2 chunks

Example chunk:
Content: Dans une architecture RAG (Retrieval-Augmented Generation), l'ingestion des données constitue la 
première étape critique pour enrichir le contexte d'un LLM avec des sources externes. 
Le rôle des loa...
Metadata: {'producer': 'macOS Version 26.5.2 (assemblage 25F84) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20260819230533Z00'00'", 'source': '../data/pdf/def.pdf', 'file_path': '../data/pdf/def.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'def', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260819230533Z00'00'", 'trapped': '', 'modDate': "D:20260819230533Z00'00'", 'creationDate': "D:20260819230533Z00'00'", 'page': 0}


[Document(metadata={'producer': 'macOS Version 26.5.2 (assemblage 25F84) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20260819230533Z00'00'", 'source': '../data/pdf/def.pdf', 'file_path': '../data/pdf/def.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'def', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260819230533Z00'00'", 'trapped': '', 'modDate': "D:20260819230533Z00'00'", 'creationDate': "D:20260819230533Z00'00'", 'page': 0}, page_content="Dans une architecture RAG (Retrieval-Augmented Generation), l'ingestion des données constitue la \npremière étape critique pour enrichir le contexte d'un LLM avec des sources externes. \nLe rôle des loaders \nLes document loaders servent de passerelle entre les formats d'origine et le pipeline de traitement. \nQu'il s'agisse de fichiers locaux (PDF, Markdown, CSV), de pages web ou de bases de données \n(Notion, SQL), le chargeur extrait le contenu brut, nettoie les données non pertinentes et standardise \nle f

### Embedding and VectorStoreDB

In [31]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid # Universally Unique Identifier
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args :
            model_name : HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    # def get_embedding_dimension(self) -> int:
    #     """Get the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded")
    #     return self.model.get_sentence_embedding_dimension()
    
# initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 6e81a331-8e9c-4b08-8013-4a68094e1454)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [37]:
class VectorStore:
    """Manages document embedding in a ChromaDB vector store"""

    def __init__(self, collection_name:str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Intialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection (where will be stored the vectors in the VDB)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Colelction: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

        #  now that we have created the collection, we need to add the documents
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store=VectorStore() # it will show 0 since we didn't add any document in the collection
vector_store

Vector store initialized. Colelction: pdf_documents
Existing documents in collection: 0


In [38]:
chunks

[Document(metadata={'producer': 'macOS Version 26.5.2 (assemblage 25F84) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20260819230533Z00'00'", 'source': '../data/pdf/def.pdf', 'file_path': '../data/pdf/def.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'def', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260819230533Z00'00'", 'trapped': '', 'modDate': "D:20260819230533Z00'00'", 'creationDate': "D:20260819230533Z00'00'", 'page': 0}, page_content="Dans une architecture RAG (Retrieval-Augmented Generation), l'ingestion des données constitue la \npremière étape critique pour enrichir le contexte d'un LLM avec des sources externes. \nLe rôle des loaders \nLes document loaders servent de passerelle entre les formats d'origine et le pipeline de traitement. \nQu'il s'agisse de fichiers locaux (PDF, Markdown, CSV), de pages web ou de bases de données \n(Notion, SQL), le chargeur extrait le contenu brut, nettoie les données non pertinentes et standardise \nle f

In [39]:
# Convert the text to embeddings
texts=[doc.page_content for doc in chunks]
texts 
# extract all the text from a partic chunk and we will generate an emb
# and we will pass it to embmanager

# Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

# Store in the VectorDataBase
vector_store.add_documents(chunks, embeddings)


Generating embeddings for 2 texts...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.87s/it]

Generated embeddings with shape: (2, 384)
Adding 2 documents to vector store...
Successfully added 2 documents to vector store
Total documents in collection: 2
